In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

warnings.filterwarnings('ignore', category=UserWarning)

os.makedirs('../../output/tables', exist_ok=True)
os.makedirs('../../output/figs', exist_ok=True)

mkdir -p failed for path C:\Users\User\.matplotlib: [WinError 5] Access is denied: 'C:\\Users\\User\\.matplotlib'
Matplotlib created a temporary cache directory at C:\Users\User\AppData\Local\Temp\matplotlib-iq167x34 because there was an issue with the default path (C:\Users\User\.matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


1. Load Data

In [2]:
df = pd.read_csv('../../output/04_Regression_handled.csv')

# Calculate Abnormal Returns (AR)
df['AR'] = df['R_i'] - (df['alpha'] + df['beta'] * df['R_m'])

Market CAAR Trajectory Plot

In [3]:
# Isolate the event window
event_data = df[(df['day'] >= -5) & (df['day'] <= 30)].copy()

# Step A: Calculate the Average Abnormal Return (AAR) for the whole market each day
aar_market = event_data.groupby('day')['AR'].mean().reset_index()
aar_market.rename(columns={'AR': 'AAR'}, inplace=True)

# Step B: Cumulatively sum the AAR to get CAAR
aar_market['CAAR'] = aar_market['AAR'].cumsum()
aar_market['CAAR(%)'] = aar_market['CAAR'] * 100

# Plotting
plt.figure(figsize=(12, 6))
sns.lineplot(x='day', y='CAAR(%)', data=aar_market, color='crimson', linewidth=2.5)

# Aesthetics
plt.axvline(x=0, color='black', linestyle='--', linewidth=1.5, label='Cyclone Landfall (Day 0)')
plt.axhline(y=0, color='gray', linestyle='-', linewidth=1)
plt.axvspan(-5, 0, color='orange', alpha=0.15, label='Anticipation Window')

plt.title('Market-Wide Cumulative Average Abnormal Return (CAAR) Trajectory', fontsize=14, fontweight='bold')
plt.xlabel('Trading Days Relative to Landfall', fontsize=12)
plt.ylabel('Cumulative Average Abnormal Return (%)', fontsize=12)
plt.legend()
plt.grid(axis='y', linestyle=':', alpha=0.7)

plt.tight_layout()
plt.savefig('../../output/figs/market_caar_trajectory.png', dpi=300)
plt.close()

Placebo test

In [4]:
# To prove our model doesn't just randomly generate negative CARs, 
# we shift the "Event Day" back by 60 trading days (approx. 3 months before the storm)
# Fake Day 0 = Actual Day -60 (A completely normal trading period in September)

df['fake_day'] = df['day'] + 60 

# Isolate the "Fake" Event Window [-5 to +30]
fake_window = df[(df['fake_day'] >= -5) & (df['fake_day'] <= 30)].copy()

# Calculate CAR for each stock during the fake window
fake_stock_cars = fake_window.groupby(['symbol', 'sector'])['AR'].sum().reset_index()
fake_stock_cars.rename(columns={'AR': 'Fake_CAR'}, inplace=True)

# Run the T-Test on the Placebo Window
placebo_results = []
sectors = fake_stock_cars['sector'].unique()

for sector in sectors:
    sector_data = fake_stock_cars[fake_stock_cars['sector'] == sector]['Fake_CAR']
    n_stocks = len(sector_data)
    
    if n_stocks > 1:
        mean_car = sector_data.mean()
        t_stat, p_val = stats.ttest_1samp(sector_data, popmean=0)
    else:
        mean_car, t_stat, p_val = np.nan, np.nan, np.nan
        
    placebo_results.append({
        'Sector': sector,
        'N': n_stocks,
        'Placebo_Mean_CAR(%)': mean_car * 100,
        'Placebo_t-stat': t_stat,
        'Placebo_p-value': p_val,
        'Significant_at_5%': p_val < 0.05 if pd.notna(p_val) else False
    })

# Market-Level Placebo Test
all_fake_cars = fake_stock_cars['Fake_CAR']
t_stat_mkt, p_val_mkt = stats.ttest_1samp(all_fake_cars, popmean=0)
placebo_results.append({
    'Sector': 'ALL STOCKS (MARKET)',
    'N': len(all_fake_cars),
    'Placebo_Mean_CAR(%)': all_fake_cars.mean() * 100,
    'Placebo_t-stat': t_stat_mkt,
    'Placebo_p-value': p_val_mkt,
    'Significant_at_5%': p_val_mkt < 0.05
})

placebo_df = pd.DataFrame(placebo_results)

# Save to CSV
placebo_df.to_csv('../../output/tables/09_placebo_test_results.csv', index=False)

# Quick check on the terminal
market_placebo = placebo_df[placebo_df['Sector'] == 'ALL STOCKS (MARKET)'].iloc[0]
print(f"\nPlacebo Market Result: CAR = {market_placebo['Placebo_Mean_CAR(%)']:.2f}% | p-value = {market_placebo['Placebo_p-value']:.4f}")
if market_placebo['Placebo_p-value'] > 0.05:
    print("SUCCESS: The placebo test is INSIGNIFICANT. This proves your real cyclone results are valid!")
else:
    print("WARNING: The placebo test is significant. The market was already behaving strangely 60 days prior.")


Placebo Market Result: CAR = 2.74% | p-value = 0.3609
SUCCESS: The placebo test is INSIGNIFICANT. This proves your real cyclone results are valid!
